In [ ]:
from pathlib import Path
import pickle
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler

import eda

In [ ]:
def load_items(stimuli_dir: Path, trial_id: int) -> np.ndarray:
    csv_path = stimuli_dir / f"civ_items_trial_{trial_id}.csv"
    df = pd.read_csv(csv_path)
    numeric = df.drop(columns=["Name"], errors="ignore")
    return numeric.to_numpy(dtype=np.int64)

def get_eda_params(items: np.ndarray) -> dict:
    n_obj = items.shape[1] - 1

    if n_obj == 3:
        n_selected = 6
        max_row_diff = 5
    elif n_obj == 5:
        n_selected = 10
        max_row_diff = 500
    else:
        raise ValueError(f"Number of objectives {n_obj} not supported")

    return {
        "n_items": items.shape[0],
        "n_obj": n_obj,
        "n_con": 1,
        "n_selected": n_selected,
        "capacity": n_selected * 10,
        "pop_size": 1_000,
        "generations": 100,
        "max_no_improve_gen": 5,
        "max_row_diff": max_row_diff,
    }

def aspi_to_p_rank(items: np.ndarray, n_obj: int, aspi_item: np.ndarray, temp: float = 0.3) -> np.ndarray:
    aspi_item = np.asarray(aspi_item, dtype=float)
    aspi_unit = aspi_item / (np.linalg.norm(aspi_item) + 1e-12)
    item_scores = items[:, :n_obj] @ aspi_unit
    ranks = item_scores.argsort().argsort().astype(float)
    scaled = ranks / (ranks.max() + 1e-12)
    logits = scaled / temp
    logits -= logits.max()
    p_rank = np.exp(logits)
    p_rank /= p_rank.sum()
    return p_rank

def run_eda_pass(items: np.ndarray, params: dict, p_rank: np.ndarray, seed: int) -> dict:
    eda_process = eda.KnapsackEDA(
        items=items,
        capacity=params["capacity"],
        n_selected=params["n_selected"],
        n_obj=params["n_obj"],
        pop_size=params["pop_size"],
        generations=params["generations"],
        max_no_improve_gen=params["max_no_improve_gen"],
        max_row_diff=params["max_row_diff"],
        seed=seed,
        p_rank=p_rank,
    )
    return eda_process.run()

def save_pass_results(run_name: str, results: dict, output_dir: Path, use_human_input: bool = True) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    result_type = "eda_human" if use_human_input else "eda"
    file_path = output_dir / f"{result_type}_{run_name}.pkl"

    with open(file_path, "wb") as f:
        pickle.dump(results, f)

    return file_path

In [ ]:
def select_test_ref(trial_id: int) -> np.ndarray:
    with open(f'card_game/eda_results/eda_trial{trial_id}.pkl', 'rb') as f:
        results = pickle.load(f)
    pf_actual = results['converged_pf_table'][-1]
    # ref_sol = np.median(pf_actual, axis=0)
    q25 = np.percentile(pf_actual, 25, axis=0)
    q75 = np.percentile(pf_actual, 75, axis=0)
    ref_sol = np.array([q25[0], q25[1], q25[2], q75[3], q75[4]])
    return ref_sol, pf_actual

def gen_aspi(ref_sol, pf_actual):
    # normalize by 95 percentile
    # q95 = np.percentile(pf_actual, 95, axis=0)
    # aspi = ref_sol/q95

    # normalize by z-score
    # aspi = (ref_sol - np.mean(pf_actual, axis=0)) / np.std(pf_actual, axis=0)

    # min-max quantile
    # q5 = np.percentile(pf_actual, 5, axis=0)
    # q95 = np.percentile(pf_actual, 95, axis=0)
    # aspi = (ref_sol - q5) / (q95 - q5 + 1e-12)

    # robust scalar (ref_sol - q50) / (q75 - q25)
    scaler = RobustScaler()
    scaler.fit(pf_actual)
    aspi = scaler.transform(ref_sol.reshape(1, -1))[0]

    return aspi

In [ ]:
# obtain item list
trial_id = 8
stimuli_dir = Path("card_game/stimuli")
items = load_items(stimuli_dir=stimuli_dir, trial_id=trial_id)

# obtain params
params = get_eda_params(items)
n_obj = params["n_obj"]
temp = 0.3

# generate aspiration vector and probabilities
ref_sol, pf_actual = select_test_ref(trial_id)
aspi = gen_aspi(ref_sol, pf_actual)
p_rank = aspi_to_p_rank(items, n_obj, aspi, temp=temp)

In [ ]:
# run EDA
run_name = "minmax_25_75"
results = run_eda_pass(
    items=items,
    params=params,
    p_rank=p_rank,
    seed=1125,
)

# save results
output_dir = Path("data/eda_results")
save_path = save_pass_results(run_name, results, output_dir, use_human_input=(p_rank is not None))
pf = results["converged_pf_table"][-1]

# save info
file_path = output_dir / f"history_{run_name}.pkl"
history = {
    "trial_id": trial_id,
    "original_aspi": ref_sol,
    "normalized_aspi": aspi,
    "temp": temp,
    "p_rank": p_rank,
}
with open(file_path, "wb") as f:
    pickle.dump(history, f)

In [ ]:
# load results
run_name = "minmax_quant_95_5"

with open(f"data/eda_results/eda_human_{run_name}.pkl", "rb") as f:
    results = pickle.load(f)
pf = results["converged_pf_table"][-1]

with open(f"data/eda_results/history_{run_name}.pkl", "rb") as f:
    history = pickle.load(f)
    ref_sol = history["original_aspi"]
    aspi = history["normalized_aspi"]

with open(f'card_game/eda_results/eda_trial{trial_id}.pkl', 'rb') as f:
        results = pickle.load(f)
pf_actual = results['converged_pf_table'][-1]

In [ ]:
# normalize pf using min-max quantile
# pf_norm = gen_aspi(pf, pf_actual)
scaler = RobustScaler()
scaler.fit(pf_actual)
pf_norm = scaler.transform(pf)

# compute center solution then normalize it using min-max quantile
center = np.median(pf, axis=0)
dist = np.linalg.norm(pf - center, axis=1) 
center_sol = pf[dist.argmin()]
# center_sol_norm = gen_aspi(center_sol, pf_actual)
center_sol_norm = scaler.transform(center_sol.reshape(1, -1))[0]

# plot normalized pf, normalized center, and aspiration 
objective_pairs = list(itertools.combinations(range(5), 2)) 
fig, axes = plt.subplots(2, 5, figsize=(24, 8))
axes = axes.ravel()
for ax, (a, b) in zip(axes, objective_pairs):
    ax.plot(pf_norm[:, a], pf_norm[:, b], "bo", alpha=0.2, markersize=3, label="PF")
    ax.plot(
        aspi[a], aspi[b],
        "ys", alpha=1, markersize=6, label="Aspiration"
    )
    ax.plot(
        center_sol_norm[a], center_sol_norm[b],
        "ks", alpha=1, markersize=6, label="Center"
    )
    ax.set_xlabel(f"Obj {a + 1}")
    ax.set_ylabel(f"Obj {b + 1}")
    # ax.set_xlim(-0.55, 1.55)
    # ax.set_ylim(-0.55, 1.55)
    # ax.set_xlim(0.3, 1.2)
    # ax.set_ylim(0.3, 1.2)
    ax.set_xlim(-2.05, 2.05)
    ax.set_ylim(-2.05, 2.05)
fig.tight_layout()
plt.show()

In [ ]:
# plot reference, center, pf, and actual pf (0-1)
pf_min = pf_actual.min(axis=0)
pf_max = pf_actual.max(axis=0)
denom = pf_max - pf_min

ref_sol_plot = (ref_sol - pf_min) / denom
center_plot = (center_sol - pf_min) / denom
pf_plot = (pf - pf_min) / denom
pf_actual_plot = (pf_actual - pf_min) / denom

fig, axes = plt.subplots(2, 5, figsize=(24, 8))
axes = axes.ravel()
for ax, (a, b) in zip(axes, objective_pairs):
    ax.plot(pf_actual_plot[:, a], pf_actual_plot[:, b], "go", alpha=0.2, markersize=3, label="PF")
    ax.plot(pf_plot[:, a], pf_plot[:, b], "bo", alpha=0.2, markersize=3, label="PF")
    ax.plot(
        ref_sol_plot[a], ref_sol_plot[b],
        "rs", alpha=1, markersize=5, label="Reference"
    )
    ax.plot(
        center_plot[a], center_plot[b],
        "ks", alpha=1, markersize=5, label="Center"
    )
    ax.set_xlabel(f"Obj {a + 1}")
    ax.set_ylabel(f"Obj {b + 1}")
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
fig.tight_layout()
plt.show()

In [ ]:
# compare pf and pf_actual
percent_objs = []
for i in range(n_obj):
    percent = np.array([100 * np.mean(pf_actual[:, i] <= x) for x in pf[:, i]])
    mean_percent = np.mean(percent)
    percent_objs.append(mean_percent)
print(percent_objs)

In [ ]:
# plot pf and reference on original scale
fig, axes = plt.subplots(2, 5, figsize=(24, 8))
axes = axes.ravel()
for ax, (a, b) in zip(axes, objective_pairs):
    ax.plot(pf_actual[:, a], pf_actual[:, b], "go", alpha=0.2, markersize=3, label="PF")
    ax.plot(pf[:, a], pf[:, b], "bo", alpha=0.2, markersize=3, label="PF")
    ax.plot(
        ref_sol[a], ref_sol[b],
        "rs", alpha=1, markersize=6, label="Reference"
    )
    ax.set_xlabel(f"Obj {a + 1}")
    ax.set_ylabel(f"Obj {b + 1}")
    ax.set_xlim(30, 150)
    ax.set_ylim(40, 150)
fig.tight_layout()
plt.show()

In [ ]:
aspi

In [ ]:
ref_sol

In [ ]:
center_sol

In [ ]:
trial_id = 8
stimuli_dir = Path("card_game/stimuli")
items = load_items(stimuli_dir=stimuli_dir, trial_id=trial_id)
n_obj = 5
temp = 0.3

aspi = np.asarray(aspi, dtype=float)
aspi_unit = aspi / (np.linalg.norm(aspi))
aspi_item_scores = items[:, :n_obj] @ aspi_unit
aspi_ranks = aspi_item_scores.argsort().argsort().astype(float)
aspi_scaled = aspi_ranks / (aspi_ranks.max() + 1e-12)
aspi_logits = aspi_scaled / temp
aspi_logits -= aspi_logits.max()
aspi_p_rank = np.exp(aspi_logits)
aspi_p_rank /= aspi_p_rank.sum()

ref_sol = np.asarray(ref_sol, dtype=float)
ref_sol_unit = ref_sol / (np.linalg.norm(ref_sol))
ref_sol_item_scores = items[:, :n_obj] @ ref_sol_unit
ref_sol_ranks = ref_sol_item_scores.argsort().argsort().astype(float)
ref_sol_scaled = ref_sol_ranks / (ref_sol_ranks.max() + 1e-12)
ref_sol_logits = ref_sol_scaled / temp
ref_sol_logits -= ref_sol_logits.max()
ref_sol_p_rank = np.exp(ref_sol_logits)
ref_sol_p_rank /= ref_sol_p_rank.sum()

In [ ]:
from scipy.stats import spearmanr, kendalltau

print(spearmanr(aspi_item_scores, ref_sol_item_scores))
print(kendalltau(aspi_item_scores, ref_sol_item_scores))

In [ ]:
print(aspi_ranks)
print(ref_sol_ranks)

In [ ]:
fig, ax = plt.subplots()
ax.plot(aspi_item_scores, ref_sol_item_scores, "bo", alpha=1, markersize=3)
ax.set_xlabel("Aspiration")
ax.set_ylabel("Reference")
plt.show()

In [ ]:
fig, ax = plt.subplots(2, 1)
ax[0].plot(aspi_p_rank)
ax[1].plot(ref_sol_p_rank)
ax[0].set_xticks(range(0, len(aspi_p_rank), 2))
ax[1].set_xticks(range(0, len(ref_sol_p_rank), 2))
ax[0].set_xlabel("Item")
ax[1].set_xlabel("Item")
ax[0].set_ylabel("Probability")
ax[1].set_ylabel("Probability")
ax[0].set_title("Aspiration")
ax[1].set_title("Reference")
fig.tight_layout()
plt.show()